# MOSUL HERITAGE — Free GPU preview session

This notebook runs **one governed, watermarked preview** with FLUX.1-schnell. It does not create a sale-ready master, mint an NFT, open an auction, or move funds. Do not add GitHub tokens, wallet keys, or signing keys to this notebook. A later GitHub Actions workflow verifies the bundle and applies the repository signing key, if configured.

## Session task contract

- **Input:** current public repository code and a fixed Mosul-inspired art prompt.
- **Output:** one PNG preview, catalog record, AI audit log, SHA-256 manifest, and a ZIP bundle.
- **Acceptance:** the image is PNG and at least 512×512; the catalog says `PUBLIC PREVIEW — NOT FOR SALE` and `Not minted`; the image hash matches the record.
- **Stop condition:** missing GPU, failed policy checks, unavailable model download, or invalid output. Do not substitute a placeholder image.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO = Path('/content/MOSUL-HERITAGE-STORE')
OUT = Path('/content/mosul-session-output')
if REPO.exists():
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch','free-gpu-session-kit','https://github.com/mosulonsol-max/MOSUL-HERITAGE-STORE.git',str(REPO)], check=True)
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Output:', OUT)

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. In Colab select Runtime > Change runtime type > GPU if available, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('This free GPU session is temporary; save the ZIP bundle before disconnecting.')

In [ ]:
# Reuse Colab's CUDA-enabled PyTorch; install only the project's GPU and base dependencies.
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','-r',str(REPO/'requirements.txt'),'-r',str(REPO/'requirements-gpu.txt')], check=True)
print('Dependencies installed.')

In [ ]:
env = os.environ.copy()
env.update({'AI_PLANNER_ENABLED':'0','ARTWORK_BACKEND':'flux-local','PYTHONPATH':str(REPO/'ai-engine')})
cmd = [sys.executable,str(REPO/'ai-engine/run_agent.py'),
       '--objective','Generate one Mosul-inspired contemporary preview; do not claim it depicts a verified historical site.',
       '--registry',str(REPO/'config/ai-capabilities.json'),
       '--audit',str(OUT/'ai-audit.jsonl'),
       '--output-dir',str(OUT),
       '--generator',str(REPO/'scripts/generate_artwork.py')]
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(REPO/'ai-engine/tests'),'-v'],cwd=REPO,env=env,check=True)
subprocess.run(cmd,cwd=REPO,env=env,check=True)
print('Generation and governed policy checks completed.')

In [ ]:
import hashlib, json, zipfile
from PIL import Image

catalog_path = OUT/'catalog.json'
catalog = json.loads(catalog_path.read_text())
if not isinstance(catalog.get('items'), list) or len(catalog['items']) != 1:
    raise RuntimeError('Expected exactly one generated preview record.')
record = catalog['items'][0]
if record.get('status') != 'PUBLIC PREVIEW — NOT FOR SALE' or record.get('mint_status') != 'Not minted':
    raise RuntimeError('Preview-only sale/mint gate failed.')
image_path = OUT / record['image']
with Image.open(image_path) as image:
    image.verify()
with Image.open(image_path) as image:
    if image.format != 'PNG' or min(image.size) < 512:
        raise RuntimeError('Image must be a valid PNG of at least 512×512.')
digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
if digest != record.get('sha256'):
    raise RuntimeError('Catalog SHA-256 does not match the image.')
manifest = {'session_type':'free_gpu_preview','preview_only':True,'image_sha256':digest,'item_id':record['id'],'generated_at':record['created_at'],'sale':'disabled','nft':'not_minted','signature':'to_be_applied_by_github_actions_if_secret_is_configured'}
(OUT/'session-manifest.json').write_text(json.dumps(manifest,indent=2) + chr(10))
bundle = Path('/content/mosul-preview-session.zip')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['catalog.json','ai-audit.jsonl','session-manifest.json',record['image']]:
        z.write(OUT/name,arcname=name)
print('Validated:', record['id'], '| SHA-256:', digest, '| bundle:', bundle)

In [ ]:
from google.colab import files
files.download('/content/mosul-preview-session.zip')

## What happens next

1. Save the downloaded `mosul-preview-session.zip`.
2. Upload it to the public repository's `artwork-data` branch as `sessions/latest.zip`.
3. Run **Publish verified Colab preview** from GitHub Actions and keep the default bundle path.
4. The workflow checks the bundle, optionally signs the preview record with the GitHub Secret, updates the preview catalog, and deploys Pages. If any check fails, it stops without publishing.

The free Colab runtime is a user-started temporary session; it does not guarantee hourly or unattended generation.